In [29]:
# This code was taken from Ryan & Matt Data Science 
import seaborn as sns
import pandas as pd

In [30]:
healthexp = sns.load_dataset('healthexp')

In [31]:
healthexp.tail()

,Year,Country,Spending_USD,Life_Expectancy
269,2020,Germany,6938.983,81.1
270,2020,France,5468.418,82.3
271,2020,Great Britain,5018.700,80.4
272,2020,Japan,4665.641,84.7
273,2020,USA,11859.179,77.0


In [32]:
# Debido a que hay una variable categórica 'Country' hay que crear nuevas variables dummies para el analisis
healthexp = pd.get_dummies(healthexp)

In [33]:
healthexp.tail()

,Year,Spending_USD,Life_Expectancy,Country_Canada,Country_France,Country_Germany,Country_Great Britain,Country_Japan,Country_USA
269,2020,6938.983,81.1,False,False,True,False,False,False
270,2020,5468.418,82.3,False,True,False,False,False,False
271,2020,5018.700,80.4,False,False,False,True,False,False
272,2020,4665.641,84.7,False,False,False,False,True,False
273,2020,11859.179,77.0,False,False,False,False,False,True


In [34]:
# Estos son todos menos 'Life_Expectancy' porque queremos predecir con base en el resto de los datos la expectativa de vida
X = healthexp.drop(['Life_Expectancy'],axis=1)

y = healthexp['Life_Expectancy']

In [35]:
from sklearn.model_selection import train_test_split

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=54)

In [37]:
from sklearn.ensemble import RandomForestRegressor


In [38]:
rfr = RandomForestRegressor(random_state=34)


In [39]:
rfr.fit(X_train,y_train)


,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",34
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [40]:
y_pred = rfr.predict(X_test)

In [41]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [42]:
mean_absolute_error(y_test,y_pred)

0.3115454545454368

In [43]:
mean_squared_error(y_test,y_pred)

0.15524932727271845

In [44]:
r2_score(y_test,y_pred)

0.9836312857564109

In [45]:
import optuna

In [46]:
from sklearn.model_selection import cross_val_score


In [47]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators',100,1000)
    max_depth = trial.suggest_int('max_depth',10,50)
    min_samples_split = trial.suggest_int('min_samples_split',2,32)
    min_samples_leaf = trial.suggest_int('min_samples_leaf',1,32)

    model = RandomForestRegressor(n_estimators=n_estimators,
                                  max_depth=max_depth,
                                  min_samples_split=min_samples_split,
                                  min_samples_leaf=min_samples_leaf)
    score = cross_val_score(model,X_train,y_train,cv=5,scoring='neg_mean_squared_error',n_jobs=-1).mean()
    return score

In [48]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.RandomSampler(seed=42))

[I 2026-06-02 21:55:35,228] A new study created in memory with name: no-name-92bb81e3-685f-4cde-a73e-c7fd040ac80c


In [49]:
study.optimize(objective,n_trials=200)

[I 2026-06-02 21:55:37,263] Trial 0 finished with value: -2.3128756957884375 and parameters: {'n_estimators': 437, 'max_depth': 48, 'min_samples_split': 24, 'min_samples_leaf': 20}. Best is trial 0 with value: -2.3128756957884375.
[I 2026-06-02 21:55:38,850] Trial 1 finished with value: -2.8993306860968433 and parameters: {'n_estimators': 240, 'max_depth': 16, 'min_samples_split': 3, 'min_samples_leaf': 28}. Best is trial 0 with value: -2.3128756957884375.
[I 2026-06-02 21:55:40,578] Trial 2 finished with value: -3.3568205475025734 and parameters: {'n_estimators': 641, 'max_depth': 39, 'min_samples_split': 2, 'min_samples_leaf': 32}. Best is trial 0 with value: -2.3128756957884375.
[I 2026-06-02 21:55:42,276] Trial 3 finished with value: -1.0176866156267157 and parameters: {'n_estimators': 850, 'max_depth': 18, 'min_samples_split': 7, 'min_samples_leaf': 6}. Best is trial 3 with value: -1.0176866156267157.
[I 2026-06-02 21:55:42,464] Trial 4 finished with value: -1.5669691458421142 and

In [50]:
study.best_params

{'n_estimators': 940,
 'max_depth': 45,
 'min_samples_split': 3,
 'min_samples_leaf': 1}

In [51]:
best_params = study.best_params

In [52]:
import matplotlib.pyplot as plt

In [53]:
optuna.visualization.plot_optimization_history(study)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'mode': 'markers',
              'name': 'Objective Value',
              'type': 'scatter',
              'x': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17,
                    18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
                    34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49,
                    50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65,
                    66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81,
                    82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97,
                    98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
                    111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122,
                    123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134,
                    135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146,
                    147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158,
                    159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170,
                    171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182,
                    183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194,
                    195, 196, 197, 198, 199],
              'y': [-2.3128756957884375, -2.8993306860968433, -3.3568205475025734,
                    -1.0176866156267157, -1.5669691458421142, -1.7883669216794122,
                    -2.2053409915133995, -1.1254780626668444, -2.6824297859660917,
                    -2.071032034206092, -3.1807387437168035, -2.1921665587526635,
                    -2.577343200387279, -3.1471609934276, -1.7006350161760817,
                    -1.8425294001889134, -2.693191592262025, -1.3889542689178322,
                    -2.5346898325732203, -0.6660948652516652, -0.5296664030492806,
                    -2.3508100199373603, -2.465671331775366, -2.143425354751674,
                    -0.5709544932289036, -2.2074416949301, -2.5802212927169954,
                    -1.0301876597534563, -2.925206453736279, -2.2268212565758154,
                    -0.6225761184710048, -2.9115099636693387, -1.3625736558780772,
                    -1.810340346456048, -3.3523231599578294, -1.5657939007866286,
                    -2.188155543661646, -1.530066336685871, -1.6074554873086249,
                    -2.5158744185747643, -2.241779832351988, -1.0390632499468082,
                    -0.9629405900464441, -1.1794301624776573, -1.3937481048123614,
                    -3.038713020034084, -2.236501628392813, -3.0353554409724604,
                    -1.8010260228852792, -2.595166012223089, -3.006959668258182,
                    -2.4065531628209635, -2.4554143280678415, -1.4630064533113787,
                    -2.902748533378481, -1.7902830321106087, -1.9592935172015626,
                    -2.1914533849447535, -2.5161477847790747, -1.1923667012486938,
                    -1.8617441033853783, -3.2697581468195898, -1.9083223112943712,
                    -2.2317737131218505, -0.8399742635091106, -2.185510785101031,
                    -2.4469003369094176, -2.666965226488302, -2.193954253028522,
                    -2.6760193437573343, -0.6547487329802154, -2.228104629021839,
                    -0.27183606800868343, -2.1889300243143586, -0.8591296606597997,
                    -2.3533919714955274, -1.6759983070237214, -0.6134275065916114,
                    -2.4332333152474934, -1.469049387809039, -1.5137941754484534,
                    -2.3161198091330446, -2.59456974904458, -0.26352476362981275,
                    -0.7421531662038978, -2.00513380195364, -0.8427268317379859,
                    -2.9231762087728677, -2.350387522284133, -2.2509942898288045,
                    -2.223581179189172, -3.028823093266374, -0.24485454519965763,
                    -1.6930969077927465, -1.6750422562880931, -3.0247015593614703,
                    -1.015066012294838, -2.0477093090891514, -3.150

In [54]:
optuna.visualization.plot_parallel_coordinate(study)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'dimensions': [{'label': 'Objective Value',
                              'range': [-3.365933482315209, -0.2038525021397634],
                              'values': [-2.3128756957884375, -2.8993306860968433,
                                         -3.3568205475025734, -1.0176866156267157,
                                         -1.5669691458421142, -1.7883669216794122,
                                         -2.2053409915133995, -1.1254780626668444,
                                         -2.6824297859660917, -2.071032034206092,
                                         -3.1807387437168035, -2.1921665587526635,
                                         -2.577343200387279, -3.1471609934276,
                                         -1.7006350161760817, -1.8425294001889134,
                                         -2.693191592262025, -1.3889542689178322,
                                         -2.5346898325732203, -0.6660948652516652,
                                         -0.5296664030492806, -2.3508100199373603,
                                         -2.465671331775366, -2.143425354751674,
                                         -0.5709544932289036, -2.2074416949301,
                                         -2.5802212927169954, -1.0301876597534563,
                                         -2.925206453736279, -2.2268212565758154,
                                         -0.6225761184710048, -2.9115099636693387,
                                         -1.3625736558780772, -1.810340346456048,
                                         -3.3523231599578294, -1.5657939007866286,
                                         -2.188155543661646, -1.530066336685871,
                                         -1.6074554873086249, -2.5158744185747643,
                                         -2.241779832351988, -1.0390632499468082,
                                         -0.9629405900464441, -1.1794301624776573,
                                         -1.3937481048123614, -3.038713020034084,
                                         -2.236501628392813, -3.0353554409724604,
                                         -1.8010260228852792, -2.595166012223089,
                                         -3.006959668258182, -2.4065531628209635,
                                         -2.4554143280678415, -1.4630064533113787,
                                         -2.902748533378481, -1.7902830321106087,
                                         -1.9592935172015626, -2.1914533849447535,
                                         -2.5161477847790747, -1.1923667012486938,
                                         -1.8617441033853783, -3.2697581468195898,
                                         -1.9083223112943712, -2.2317737131218505,
                                         -0.8399742635091106, -2.185510785101031,
                                         -2.4469003369094176, -2.666965226488302,
                                         -2.193954253028522, -2.6760193437573343,
                                         -0.6547487329802154, -2.228104629021839,
                                         -0.27183606800868343, -2.1889300243143586,
                                         -0.8591296606597997, -2.3533919714955274,
                                         -1.6759983070237214, -0.6134275065916114,
                                         -2.4332333152474934, -1.469049387809039,
                                         -1.5137941754484534, -2.3161198091330446,
                                         -2.59456974904458, -0.26352476362981275,
                                         -0.7421531662038978, -2.00513380195364,
                                         -0.8427268317379859, -2.9231762087728677,
                                         -2.350387522284133, -2.2509942898288045,
                                         -2.223581179189172, -3.028823093266374,
                                         -0.24485454519965763, -

In [55]:
optuna.visualization.plot_slice(study, params=['n_estimators', 'max_depth', 'min_samples_leaf', 'min_samples_split'])

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'marker': {'color': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
                                   14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25,
                                   26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37,
                                   38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49,
                                   50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61,
                                   62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73,
                                   74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85,
                                   86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97,
                                   98, 99, 100, 101, 102, 103, 104, 105, 106, 107,
                                   108, 109, 110, 111, 112, 113, 114, 115, 116,
                                   117, 118, 119, 120, 121, 122, 123, 124, 125,
                                   126, 127, 128, 129, 130, 131, 132, 133, 134,
                                   135, 136, 137, 138, 139, 140, 141, 142, 143,
                                   144, 145, 146, 147, 148, 149, 150, 151, 152,
                                   153, 154, 155, 156, 157, 158, 159, 160, 161,
                                   162, 163, 164, 165, 166, 167, 168, 169, 170,
                                   171, 172, 173, 174, 175, 176, 177, 178, 179,
                                   180, 181, 182, 183, 184, 185, 186, 187, 188,
                                   189, 190, 191, 192, 193, 194, 195, 196, 197,
                                   198, 199],
                         'colorbar': {'title': {'text': 'Trial'}, 'x': 1.0, 'xpad': 40},
                         'colorscale': [[0.0, 'rgb(247,251,255)'], [0.125,
                                        'rgb(222,235,247)'], [0.25,
                                        'rgb(198,219,239)'], [0.375,
                                        'rgb(158,202,225)'], [0.5,
                                        'rgb(107,174,214)'], [0.625,
                                        'rgb(66,146,198)'], [0.75,
                                        'rgb(33,113,181)'], [0.875,
                                        'rgb(8,81,156)'], [1.0, 'rgb(8,48,107)']],
                         'line': {'color': 'Grey', 'width': 0.5},
                         'showscale': True},
              'mode': 'markers',
              'name': 'Feasible Trial',
              'showlegend': False,
              'type': 'scatter',
              'x': [48, 16, 39, 18, 31, 15, 42, 11, 48, 14, 30, 37, 17, 46, 18,
                    21, 32, 50, 43, 13, 35, 23, 29, 33, 27, 36, 20, 13, 43, 17, 46,
                    27, 30, 23, 38, 20, 11, 21, 30, 41, 35, 44, 34, 19, 25, 14, 37,
                    19, 35, 46, 13, 10, 16, 19, 40, 33, 20, 35, 30, 10, 49, 48, 44,
                    22, 38, 50, 40, 22, 47, 36, 23, 11, 34, 24, 18, 31, 50, 21, 49,
                    17, 39, 40, 20, 14, 38, 29, 35, 35, 16, 34, 36, 25, 12, 38, 10,
                    38, 42, 43, 41, 27, 47, 46, 13, 33, 23, 42, 32, 14, 14, 43, 50,
                    48, 13, 47, 29, 36, 25, 49, 46, 32, 38, 46, 46, 17, 13, 31, 16,
                    39, 45, 43, 25, 29, 45, 32, 20, 34, 24, 41, 32, 11, 18, 34, 24,
                    35, 47, 50, 19, 16, 24, 17, 21, 28, 38, 13, 21, 18, 25, 20, 31,
                    19, 23, 46, 30, 40, 24, 50, 16, 21, 31, 26, 20, 32, 17, 19, 33,
                    12, 50, 37, 26, 49, 17, 44, 16, 31, 43, 22, 46, 35, 23, 31, 22,
                    33, 43, 26, 39, 32, 47, 48, 35, 33],
              'xaxis': 'x',
              'y': [-2.3128756957884375, -2.8993306860968433, -3.3568205475025734,
                    -1.0176866156267157, -1.5669691458421142, -1.7883669216794122,
                    -2.2053409915133995, -1.1254780626668444, -2.6824297859660917,
                    -2.071032034206092, -3.1807387437168035, -2.1921665587526635,
                    -2.577343200387279, -3.1471609934276, -1.700635016176

In [56]:
optuna.visualization.plot_param_importances(study)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'cliponaxis': False,
              'hovertemplate': [max_depth (IntDistribution):
                                5.872648154647039e-05<extra></extra>, n_estimators
                                (IntDistribution):
                                8.313489294584116e-05<extra></extra>,
                                min_samples_split (IntDistribution):
                                0.002992332396614846<extra></extra>,
                                min_samples_leaf (IntDistribution):
                                0.996865806228893<extra></extra>],
              'name': 'Objective Value',
              'orientation': 'h',
              'text': [<0.01, <0.01, <0.01, 1.00],
              'textposition': 'outside',
              'type': 'bar',
              'x': [5.872648154647039e-05, 8.313489294584116e-05,
                    0.002992332396614846, 0.996865806228893],
              'y': [max_depth, n_estimators, min_samples_split, min_samples_leaf]}],
    'layout': {'template': '...',
               'title': {'text': 'Hyperparameter Importances'},
               'xaxis': {'title': {'text': 'Hyperparameter Importance'}},
               'yaxis': {'title': {'text': 'Hyperparameter'}}}
})

In [57]:
best_n_estimators = best_params['n_estimators']
best_max_depth = best_params['max_depth']
best_min_samples_split = best_params['min_samples_split']
best_min_samples_leaf = best_params['min_samples_leaf']

In [58]:
best_model = RandomForestRegressor(n_estimators=best_n_estimators, max_depth=best_max_depth, min_samples_split=best_min_samples_split, min_samples_leaf=best_min_samples_leaf)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

In [60]:
mean_absolute_error(y_test, y_pred)

0.3230935517791243

In [61]:
mean_squared_error(y_test, y_pred)

0.16319614849404615

In [62]:
r2_score(y_test, y_pred)

0.9827934125881214